In [1]:
from HINGE.evaluation_synthetic import run_synthetic_evaluation_with_dropout

from HINGE.feature_mapping import prepare_expression_features

from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

In [2]:
SAVE_DIR = Path('./hinge_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
with open(SAVE_DIR / 'checkpoint_step1.pkl', "rb") as f:
    data_dict=pickle.load(f)

data_dict

{'RNA_panel1': {'s1': AnnData object with n_obs × n_vars = 6989 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's2': AnnData object with n_obs × n_vars = 8978 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's3': AnnData object with n_obs × n_vars = 449221 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's4': AnnData object with n_obs × n_vars = 7442 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's5': AnnData object with n_obs × n_vars = 16540 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'f

## load celltype annotation

In [5]:
# ═══════════════════════════════════════════════════════════════
# LOAD ANNOTATIONS
# ═══════════════════════════════════════════════════════════════

ct_480 = pd.read_csv('./480_cell_annotations.csv')   # columns: cell_id, cell_type
ct_5k  = pd.read_csv('./5k_cell_annotations.csv')


In [6]:
panel_480_xenium_to_celltype_dict = {
    "s1": "output-XETG00288__0064841__15236-1-16A__20260409__150232",
    "s2": "output-XETG00288__0064841__15281-1-16A__20260409__150232",
    "s3": "output-XETG00288__0064841__HS44__20260409__150232",
    "s4": "output-XETG00288__0104483__15037-1-16A__20260409__150231",
    "s5": "output-XETG00288__0104483__15077-1-16A__20260409__150232",
    "s6": "output-XETG00288__0104483__HV2__20260409__150232"
}

panel_5k_xenium_to_celltype_dict = {
    "s7": "output-XETG00288__0064839__15236-1-16A__20260327__155146",
    "s8": "output-XETG00288__0064839__15281-1-16B__20260327__155146",
    "s9": "output-XETG00288__0064839__HS44__20260327__155146",
    "s10": "output-XETG00288__0081614__15037-1-16A__20260327__155145",
    "s11": "output-XETG00288__0081614__15077-1-16A__20260327__155146",
    "s12": "output-XETG00288__0081614__HV2__20260327__155146"
}

In [7]:
import cv2





def transform_annotation_df_to_he(
    annot_df,
    xenium_to_transform_dict,
    transformation_matrix_base_dir,
    annot_sample_col="slides",
    x_col="x",
    y_col="y",
    pixel_size=0.2125,
    out_x_col="x_he",
    out_y_col="y_he",
):
    """
    Transform annotation coordinates from Xenium space to H&E space
    using the same homography used for RNA data.

    Parameters
    ----------
    annot_df : pd.DataFrame
        Must contain sample column and x/y coordinate columns.
    xenium_to_transform_dict : dict
        Maps Xenium sample name -> transformation folder name
    transformation_matrix_base_dir : str or Path
        Base directory containing transformation folders
    annot_sample_col : str
        Column in annot_df with Xenium sample names
    x_col, y_col : str
        Original Xenium-space coordinates
    pixel_size : float
        Xenium micron-to-pixel conversion factor
    out_x_col, out_y_col : str
        Names of output transformed coordinate columns

    Returns
    -------
    annot_df_out : pd.DataFrame
        Copy of input df with added transformed coordinates
    """
    annot_df_out = annot_df.copy()
    annot_df_out[out_x_col] = np.nan
    annot_df_out[out_y_col] = np.nan

    for xenium_name, transform_name in xenium_to_transform_dict.items():
        mask = annot_df_out[annot_sample_col] == xenium_name
        if mask.sum() == 0:
            print(f"No annotation rows found for sample: {xenium_name}")
            continue

        transformation_matrix_path = os.path.join(
            transformation_matrix_base_dir,
            transform_name,
            "dapi_to_he_homography_level0.csv"
        )

        H_dapi_to_he = np.loadtxt(transformation_matrix_path, delimiter=",")

        sub = annot_df_out.loc[mask, [x_col, y_col]].copy()

        # same conversion used for RNA data
        x_px = sub[x_col].to_numpy() / pixel_size
        y_px = sub[y_col].to_numpy() / pixel_size

        pts = np.column_stack([x_px, y_px]).astype(np.float32).reshape(-1, 1, 2)
        pts_he = cv2.perspectiveTransform(pts, H_dapi_to_he.astype(np.float64))

        annot_df_out.loc[mask, out_x_col] = pts_he[:, 0, 0]
        annot_df_out.loc[mask, out_y_col] = pts_he[:, 0, 1]

        print(f"{xenium_name}: transformed {mask.sum()} annotation rows")

    return annot_df_out

In [8]:
transformation_matrix_base_dir = Path("/home/yyamamo2/lupus_project/Xenium_skin_COSIE/cosie-main/7_31_2026_skin_xenium_transformation")

In [9]:

panel_480_xenium_to_transformation_dict = {

"output-XETG00288__0064841__15236-1-16A__20260409__150232": "480_15236-1-16A",
"output-XETG00288__0064841__15281-1-16A__20260409__150232":"480_15281-1-16A",
"output-XETG00288__0064841__HS44__20260409__150232":"480_HS44",
"output-XETG00288__0104483__15037-1-16A__20260409__150231":"480_15037-1-16A",
"output-XETG00288__0104483__15077-1-16A__20260409__150232":"480_15077-1-16A",
"output-XETG00288__0104483__HV2__20260409__150232":"480_HV2"



    
}

ct_480_he = transform_annotation_df_to_he(
    annot_df=ct_480,
    xenium_to_transform_dict=panel_480_xenium_to_transformation_dict,
    transformation_matrix_base_dir=transformation_matrix_base_dir,
    annot_sample_col="slides",
    x_col="x",
    y_col="y",
    pixel_size=0.2125,
    out_x_col="x_he",
    out_y_col="y_he",
)



output-XETG00288__0064841__15236-1-16A__20260409__150232: transformed 6007 annotation rows
output-XETG00288__0064841__15281-1-16A__20260409__150232: transformed 8055 annotation rows
output-XETG00288__0064841__HS44__20260409__150232: transformed 422735 annotation rows
output-XETG00288__0104483__15037-1-16A__20260409__150231: transformed 6296 annotation rows
output-XETG00288__0104483__15077-1-16A__20260409__150232: transformed 13028 annotation rows
output-XETG00288__0104483__HV2__20260409__150232: transformed 20617 annotation rows


In [10]:
transformation_matrix_base_dir = Path("/home/yyamamo2/lupus_project/Xenium_skin_COSIE/cosie-main/7_31_2026_skin_xenium_transformation")
panel_5k_xenium_to_transformation_dict = {

"output-XETG00288__0064839__15236-1-16A__20260327__155146": "5k_15236-1-16A",
"output-XETG00288__0064839__15281-1-16B__20260327__155146":"5k_15281-1-16B",
"output-XETG00288__0064839__HS44__20260327__155146":"5k_HS44",
"output-XETG00288__0081614__15037-1-16A__20260327__155145":"5k_15037-1-16A",
"output-XETG00288__0081614__15077-1-16A__20260327__155146":"5k_15077-1-16A",
"output-XETG00288__0081614__HV2__20260327__155146":"5k_HV2"



    
}

ct_5k_he = transform_annotation_df_to_he(
    annot_df=ct_5k,
    xenium_to_transform_dict=panel_5k_xenium_to_transformation_dict,
    transformation_matrix_base_dir=transformation_matrix_base_dir,
    annot_sample_col="slides",
    x_col="x",
    y_col="y",
    pixel_size=0.2125,
    out_x_col="x_he",
    out_y_col="y_he",
)



output-XETG00288__0064839__15236-1-16A__20260327__155146: transformed 6042 annotation rows
output-XETG00288__0064839__15281-1-16B__20260327__155146: transformed 7730 annotation rows
output-XETG00288__0064839__HS44__20260327__155146: transformed 377186 annotation rows
output-XETG00288__0081614__15037-1-16A__20260327__155145: transformed 6506 annotation rows
output-XETG00288__0081614__15077-1-16A__20260327__155146: transformed 13158 annotation rows
output-XETG00288__0081614__HV2__20260327__155146: transformed 23373 annotation rows


In [11]:
sample_map = {}
sample_map.update(panel_480_xenium_to_celltype_dict)
sample_map.update(panel_5k_xenium_to_celltype_dict)


In [12]:
annot_df_he = pd.concat([ct_480_he, ct_5k_he], ignore_index=True)

In [13]:
from scipy.spatial import cKDTree

def add_celltypes_by_spatial_match(
    predicted_rna,
    annot_df,
    sample_map,
    annot_sample_col="slides",
    celltype_col="cell_type",
    x_col="x",
    y_col="y",
    spatial_key="spatial",
    max_dist=10.0,
):
    for key, adata in predicted_rna.items():
        if key not in sample_map:
            print(f"{key}: not found in sample_map, skipping")
            continue

        sample_name = sample_map[key]

        sub = annot_df[annot_df[annot_sample_col] == sample_name].copy()

        if len(sub) == 0:
            print(f"{key}: no annotations found for sample '{sample_name}'")
            adata.obs["cell_type"] = "Unknown_lowquality"
            adata.obs["celltype_annotation_dist"] = np.nan
            continue

        annot_xy = sub[[x_col, y_col]].to_numpy()
        annot_labels = sub[celltype_col].to_numpy()

        pred_xy = adata.obsm[spatial_key]

        tree = cKDTree(annot_xy)
        dist, idx = tree.query(pred_xy, k=1)

        matched_labels = annot_labels[idx].astype(object)
        matched_labels[dist > max_dist] = "Unknown"

        adata.obs["cell_type"] = matched_labels
        adata.obs["matched_annotation_dist"] = dist

        print(
            f"{key}: sample={sample_name}, "
            f"n_pred={adata.n_obs}, n_annot={len(sub)}, "
            f"assigned={(dist <= max_dist).sum()} "
            f"({(dist <= max_dist).mean():.1%})"
        )
        
        
        
add_celltypes_by_spatial_match(
    predicted_rna=data_dict['RNA_panel1'],
    annot_df=annot_df_he,
    sample_map=sample_map,
    annot_sample_col="slides",
    celltype_col="cell_type",
    x_col="x_he",
    y_col="y_he",
    spatial_key="spatial",
    max_dist=5.0,
)

add_celltypes_by_spatial_match(
    predicted_rna=data_dict['RNA_panel1'],
    annot_df=annot_df_he,
    sample_map=sample_map,
    annot_sample_col="slides",
    celltype_col="cell_type",
    x_col="x_he",
    y_col="y_he",
    spatial_key="spatial",
    max_dist=5.0,
)

s1: sample=output-XETG00288__0064841__15236-1-16A__20260409__150232, n_pred=6989, n_annot=6007, assigned=6007 (85.9%)
s2: sample=output-XETG00288__0064841__15281-1-16A__20260409__150232, n_pred=8978, n_annot=8055, assigned=8055 (89.7%)
s3: sample=output-XETG00288__0064841__HS44__20260409__150232, n_pred=449221, n_annot=422735, assigned=422737 (94.1%)
s4: sample=output-XETG00288__0104483__15037-1-16A__20260409__150231, n_pred=7442, n_annot=6296, assigned=6296 (84.6%)
s5: sample=output-XETG00288__0104483__15077-1-16A__20260409__150232, n_pred=16540, n_annot=13028, assigned=13028 (78.8%)
s6: sample=output-XETG00288__0104483__HV2__20260409__150232, n_pred=24862, n_annot=20617, assigned=20617 (82.9%)
s1: sample=output-XETG00288__0064841__15236-1-16A__20260409__150232, n_pred=6989, n_annot=6007, assigned=6007 (85.9%)
s2: sample=output-XETG00288__0064841__15281-1-16A__20260409__150232, n_pred=8978, n_annot=8055, assigned=8055 (89.7%)
s3: sample=output-XETG00288__0064841__HS44__20260409__15023

# load individual sample

In [14]:
#select which sample to use for the mapping

uni_480_list = [data_dict['HE_panel1'][1]]  # only the first sample for demonstration
uni_5k_list = [data_dict['HE_panel2'][1]]  # only the first sample for demonstration
rna_480_list = [data_dict['RNA_panel1']['s2']]  # only the first sample for demonstration
rna_5k_list = [data_dict['RNA_panel2']['s8']]  # only the first sample for demonstration

In [15]:
uni_480_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [16]:
rna_480_list

[AnnData object with n_obs × n_vars = 8978 × 5101
     obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
     var: 'gene_ids', 'feature_types', 'genome'
     obsm: 'spatial']

In [17]:

uni_480_normalized_list = []

for adata in uni_480_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_480_normalized_list.append(adata_norm)

In [18]:
uni_5k_normalized_list = []

for adata in uni_5k_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_5k_normalized_list.append(adata_norm)

In [19]:
uni_480_normalized_list

[AnnData object with n_obs × n_vars = 8978 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [20]:
adata_panel1          = rna_480_list[0]
uni_panel1_normed     = uni_480_normalized_list[0]
adata_panel2          = rna_5k_list[0]
uni_panel2_normed     = uni_5k_normalized_list[0]

# run test

In [22]:
df_accuracy, df_celltype, fakes, stats = run_synthetic_evaluation_with_dropout(
    adata_panel1          = adata_panel1,
    uni_panel1_normed     = uni_panel1_normed,
    adata_panel2          = adata_panel2,
    uni_panel2_normed     = uni_panel2_normed,
    alpha_list            = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75],
    n_seeds_per_alpha     = 2,
    MAX_DIST              = None,
    dropout_frac          = 0.10,
    dropout_mode          = "independent",   # or "symmetric"
    SAMPLE_NAME           = "s1",

    N_PCA_MORPH       = 2048,
    n_gene_comps = 512,


    
    evaluate_hinge = True,
    
    leiden_resolution     = 0.5,         # tune if needed
    min_cells_per_cluster = 20,
    
    evaluate_celltype = True,
    celltype_key      = "cell_type",    # must exist in adata_panel1.obs
    
    run_hinge1 = True,
    run_hinge2 = False,
    run_hinge3 = False,
    run_hinge4 = False,
    run_hinge5 = True,
    run_hinge6 = True,           # ← new
)

  Computing population-level difference statistics
  Panel1: 8978 cells  |  Panel2: 8899 cells

  Coordinate displacement field:
    mean dx          = -3.26 um
    mean dy          = -1.91 um
    mean NN dist     = 39.32 um
    kept fraction    = 100.0%

  UNI (normalized space, 2048D):
    |delta_mean| mean = 0.00417
    delta_std    mean = 0.01717

  Expression (444 genes):
    |delta_mean| mean = 0.0782
    delta_std    mean = 0.1463
  Created s1_t: alpha=0.50  seed=0  xy_method=displacement_field  mean coord shift=21.46 um  n=8978
  Created s1_t: alpha=0.50  seed=1  xy_method=displacement_field  mean coord shift=21.48 um  n=8978
  Created s1_t: alpha=0.75  seed=2  xy_method=displacement_field  mean coord shift=30.65 um  n=8978
  Created s1_t: alpha=0.75  seed=3  xy_method=displacement_field  mean coord shift=30.57 um  n=8978
  Created s1_t: alpha=1.00  seed=4  xy_method=displacement_field  mean coord shift=40.25 um  n=8978
  Created s1_t: alpha=1.00  seed=5  xy_method=displacement